# Pipeline v4 — Au-delà du plafond classique

## Stratégies pour dépasser 66%

| Levier | Gain estimé | Mécanisme |
|--------|-------------|----------|
| `class_weight='balanced'` | +2–4% recall cohérent | Pénalise mieux les faux négatifs |
| Ajustement de seuil (0.5 → ~0.38) | +1–3% F1 | Compense le biais vers 'incoherent' |
| Stacking (RF + LR + SVM) | +2–4% stable | Diversité de modèles |
| CLIP / SentenceTransformers | +5–12% | Features sémantiques |
| Augmentation données texte | +1–3% | Paraphrase légère |

**Ordre d'exécution recommandé :** Cell 1→13 classique amélioré, puis Cell 14→17 CLIP si disponible.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 — Dépendances
# ═══════════════════════════════════════════════════════════════════
!pip install opencv-python scikit-image vaderSentiment joblib sentence-transformers --quiet
# Pour CLIP (optionnel mais fortement recommandé) :
# !pip install git+https://github.com/openai/CLIP.git --quiet
# !pip install torch torchvision --quiet

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — Imports
# ═══════════════════════════════════════════════════════════════════
import os, warnings, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray
from skimage import io, transform, filters
from skimage.filters import threshold_otsu, threshold_local, scharr
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
from scipy.stats import entropy as scipy_entropy, pearsonr
from scipy.spatial.distance import cosine, cityblock, chebyshev, canberra, braycurtis
from skimage.measure import regionprops

from joblib import Parallel, delayed, Memory
import multiprocessing
N_JOBS = max(1, multiprocessing.cpu_count() - 1)

CACHE_DIR = './joblib_cache_v4'
memory = Memory(CACHE_DIR, verbose=0)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cross_decomposition import CCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_OK = True
    vader = SentimentIntensityAnalyzer()
except ImportError:
    VADER_OK = False

warnings.filterwarnings('ignore')
for r in ['punkt','stopwords','wordnet','omw-1.4',
          'averaged_perceptron_tagger_eng','punkt_tab']:
    nltk.download(r, quiet=True)

lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words('english'))

# ── Détection CLIP / SentenceTransformers ──────────────────────────
CLIP_AVAILABLE = False
SBERT_AVAILABLE = False
try:
    import clip, torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    CLIP_MODEL, CLIP_PREPROCESS = clip.load('ViT-B/32', device=DEVICE)
    CLIP_AVAILABLE = True
    print(f'✓ CLIP disponible ({DEVICE})')
except Exception as e:
    print(f'✗ CLIP non disponible: {e}')

try:
    from sentence_transformers import SentenceTransformer
    SBERT_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
    SBERT_AVAILABLE = True
    print('✓ SentenceTransformer disponible')
except Exception as e:
    print(f'✗ SentenceTransformer non disponible: {e}')

print(f'\nParallélisme : {N_JOBS} workers')
print('Imports OK — Pipeline v4 Advanced')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Chargement données
# ═══════════════════════════════════════════════════════════════════
DATA_DIR = '../data/processed'

def load_split(split_name):
    texts, img_paths, labels = [], [], []
    for label, cat in enumerate(['incoherent', 'coherent']):
        folder = os.path.join(DATA_DIR, split_name, cat)
        if not os.path.exists(folder):
            print(f'Dossier {folder} non trouvé. Skipping.')
            continue
        for f in sorted(os.listdir(folder)):
            if f.endswith('.txt'):
                with open(os.path.join(folder, f), 'r', encoding='utf-8') as fh:
                    texts.append(fh.read().strip())
                img_paths.append(os.path.join(folder, f.replace('.txt', '.jpg')))
                labels.append(label)
    texts, img_paths, labels = shuffle(texts, img_paths,
                                       np.array(labels), random_state=42)
    return np.array(texts), np.array(img_paths), labels

print('Chargement des splits...')
t_train, p_train, y_train = load_split('train')
t_val,   p_val,   y_val   = load_split('validation')
t_test,  p_test,  y_test  = load_split('test')

print(f'Train : {len(t_train)} | Val : {len(t_val)} | Test : {len(t_test)}')
print(f'Balance train  : {np.bincount(y_train)}')
print(f'Balance val    : {np.bincount(y_val)}')
print(f'Balance test   : {np.bincount(y_test)}')

# Class weights pour équilibrer
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
CLASS_WEIGHT = {0: cw[0], 1: cw[1]}
print(f'\nClass weights : {CLASS_WEIGHT}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — Features image spatiales (identique v3, mis en cache)
# ═══════════════════════════════════════════════════════════════════
IMG_SIZE   = (192, 192)
GRID_N     = 3
HOG_PIXELS = 8
LBP_RADIUS = 2
LBP_POINTS = 8 * LBP_RADIUS
N_HSV_BINS = 16

def load_image(img_path):
    img = io.imread(img_path)
    if img.ndim == 2:     img = np.stack([img]*3, axis=-1)
    if img.shape[2] == 4: img = img[:,:,:3]
    img  = (transform.resize(img, IMG_SIZE, anti_aliasing=True) * 255).astype(np.uint8)
    gray = rgb2gray(img)
    hsv  = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    return img, gray, hsv

def segment_canny(gray):   return cv2.Canny((gray*255).astype(np.uint8), 50, 150).astype(float)/255.0
def segment_deriche(gray): mag=scharr(gray); return (mag-mag.min())/(mag.max()-mag.min()+1e-8)
def segment_otsu(gray):    return (gray > threshold_otsu(gray)).astype(float)
def segment_adaptive(gray):
    g8=( gray*255).astype(np.uint8)
    block=min(35,(min(gray.shape[:2])//4)|1)
    return (g8 > threshold_local(g8,block_size=block,offset=10)).astype(float)

def compute_saliency_map(gray):
    g8=( gray*255).astype(np.float32)
    fft=np.fft.fft2(g8)
    log_amp=np.log(np.abs(fft)+1e-8)
    smooth=cv2.filter2D(log_amp,-1,np.ones((3,3),np.float32)/9)
    sr=log_amp-smooth
    rec=np.fft.ifft2(np.exp(sr+1j*np.angle(fft))).real
    sal=cv2.GaussianBlur(rec**2,(9,9),2.5)
    return (sal-sal.min())/(sal.max()-sal.min()+1e-8)

def features_per_zone(zone_gray, zone_rgb, zone_hsv):
    hog_f=hog(zone_gray,orientations=8,pixels_per_cell=(HOG_PIXELS,HOG_PIXELS),
              cells_per_block=(2,2),feature_vector=True)
    h_h=np.histogram(zone_hsv[:,:,0],bins=N_HSV_BINS,range=(0,180))[0].astype(float)
    s_h=np.histogram(zone_hsv[:,:,1],bins=N_HSV_BINS,range=(0,256))[0].astype(float)
    v_h=np.histogram(zone_hsv[:,:,2],bins=N_HSV_BINS,range=(0,256))[0].astype(float)
    hsv_f=np.concatenate([h_h,s_h,v_h]); hsv_f/=(hsv_f.sum()+1e-8)
    lbp=local_binary_pattern(zone_gray,LBP_POINTS,LBP_RADIUS,method='uniform')
    lbp_f=np.histogram(lbp,bins=LBP_POINTS+2,range=(0,LBP_POINTS+2))[0].astype(float)
    lbp_f/=(lbp_f.sum()+1e-8)
    stats=np.array([zone_gray.mean(),zone_gray.std(),
                    zone_rgb.mean(axis=(0,1)).mean(),zone_hsv[:,:,1].mean()])
    return np.concatenate([hog_f,hsv_f,lbp_f,stats])

@memory.cache
def extract_spatial_features(img_path):
    try:
        img, gray, hsv = load_image(img_path)
        h, w = img.shape[:2]
        zh, zw = h//GRID_N, w//GRID_N
        half_h, half_w = h//2, w//2

        zone_feats=[]
        for r in range(GRID_N):
            for c in range(GRID_N):
                zone_feats.append(features_per_zone(
                    gray[r*zh:(r+1)*zh, c*zw:(c+1)*zw],
                    img [r*zh:(r+1)*zh, c*zw:(c+1)*zw],
                    hsv [r*zh:(r+1)*zh, c*zw:(c+1)*zw]))
        zone_vector=np.concatenate(zone_feats)

        canny_map=segment_canny(gray); deriche_map=segment_deriche(gray)
        adapt_map=segment_adaptive(gray); otsu_map=segment_otsu(gray)

        def zone_means(arr):
            return arr[:GRID_N*zh,:GRID_N*zw].reshape(GRID_N,zh,GRID_N,zw).mean(axis=(1,3)).ravel()

        sal_map=compute_saliency_map(gray)
        sal_grid=zone_means(sal_map)
        sal_entropy=scipy_entropy(sal_grid+1e-8)

        canny_coords=np.argwhere(canny_map>0.5)
        if len(canny_coords)>0:
            cy_norm=canny_coords[:,0].mean()/h; cx_norm=canny_coords[:,1].mean()/w
            cy_std=canny_coords[:,0].std()/h;   cx_std=canny_coords[:,1].std()/w
        else:
            cy_norm=cx_norm=0.5; cy_std=cx_std=0.0

        asym_lr_canny=canny_map[:,:half_w].mean()-canny_map[:,half_w:].mean()
        asym_tb_canny=canny_map[:half_h,:].mean()-canny_map[half_h:,:].mean()
        asym_lr_sal=sal_map[:,:half_w].mean()-sal_map[:,half_w:].mean()
        asym_tb_sal=sal_map[:half_h,:].mean()-sal_map[half_h:,:].mean()
        top_hue=hsv[:half_h,:,0].mean()/180.0; bot_hue=hsv[half_h:,:,0].mean()/180.0
        top_sat=hsv[:half_h,:,1].mean()/255.0; bot_sat=hsv[half_h:,:,1].mean()/255.0
        top_val=hsv[:half_h,:,2].mean()/255.0; bot_val=hsv[half_h:,:,2].mean()/255.0

        try:
            binary=(otsu_map>0); dist=ndi.distance_transform_edt(binary)
            min_d=max(5,min(gray.shape)//10)
            coords=peak_local_max(dist,min_distance=min_d,labels=binary)
            mask=np.zeros(dist.shape,dtype=bool); mask[tuple(coords.T)]=True
            markers,_=ndi.label(mask)
            ws_map=watershed(-dist,markers,mask=binary)
            n_regions=ws_map.max()
            regs=regionprops(ws_map)
            if regs:
                areas=[r.area for r in regs]
                ws_area_mean=np.mean(areas)/(h*w); ws_area_std=np.std(areas)/(h*w)
                cy_ws=np.mean([r.centroid[0]/h for r in regs])
                cx_ws=np.mean([r.centroid[1]/w for r in regs])
                cy_ws_std=np.std([r.centroid[0]/h for r in regs])
                cx_ws_std=np.std([r.centroid[1]/w for r in regs])
            else:
                ws_area_mean=ws_area_std=0.0; cy_ws=cx_ws=0.5; cy_ws_std=cx_ws_std=0.0
        except Exception:
            n_regions=1; ws_area_mean=ws_area_std=0.0; cy_ws=cx_ws=0.5; cy_ws_std=cx_ws_std=0.0

        hog_global=hog(gray,orientations=9,pixels_per_cell=(16,16),
                       cells_per_block=(2,2),feature_vector=True)
        spatial_scalars=np.array([
            cy_norm,cx_norm,cy_std,cx_std,
            asym_lr_canny,asym_tb_canny,asym_lr_sal,asym_tb_sal,
            asym_hue:=top_hue-bot_hue, top_sat-bot_sat, top_val-bot_val,
            top_hue,bot_hue,top_sat,bot_sat,top_val,bot_val,
            n_regions/50.0,ws_area_mean,ws_area_std,cy_ws,cx_ws,cy_ws_std,cx_ws_std,
            sal_entropy,sal_map.mean(),sal_map.std(),sal_map.max(),
            canny_map.mean(),otsu_map.mean(),
            scipy_entropy(np.histogram(gray.flatten(),bins=32)[0].astype(float)+1e-8),
            gray.mean(),gray.std(),
        ])
        return np.concatenate([zone_vector,zone_means(canny_map),zone_means(deriche_map),
                                zone_means(adapt_map),zone_means(otsu_map),sal_grid,
                                spatial_scalars,hog_global])
    except Exception as e:
        print(f'Erreur {img_path}: {e}')
        return np.zeros(500)

def extract_all_images(paths, desc=''):
    t0=time.time()
    out=Parallel(n_jobs=N_JOBS,prefer='processes')(
        delayed(extract_spatial_features)(p) for p in paths)
    print(f'  {desc}: {time.time()-t0:.1f}s → shape {np.array(out).shape}')
    return np.array(out)

print('Extraction features image...')
F_img_train = extract_all_images(p_train, 'train')
F_img_val   = extract_all_images(p_val,   'val')
F_img_test  = extract_all_images(p_test,  'test')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 — Features texte : TF-IDF + stats spatiales
# ═══════════════════════════════════════════════════════════════════
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens
                    if t.isalpha() and t not in STOP)

print('TF-IDF...')
t_train_clean = [preprocess_text(t) for t in t_train]
t_val_clean   = [preprocess_text(t) for t in t_val]
t_test_clean  = [preprocess_text(t) for t in t_test]

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2),
                         sublinear_tf=True, min_df=2)
T_tfidf_train = tfidf.fit_transform(t_train_clean).toarray()
T_tfidf_val   = tfidf.transform(t_val_clean).toarray()
T_tfidf_test  = tfidf.transform(t_test_clean).toarray()
print(f'TF-IDF shape: {T_tfidf_train.shape}')

SPATIAL_KEYWORDS = {
    'top':    ['top','upper','above','sky','ceiling','cloud'],
    'bottom': ['bottom','lower','below','ground','floor','base'],
    'left':   ['left','western','west'],
    'right':  ['right','eastern','east'],
    'center': ['center','centre','middle','central'],
    'nature': ['sky','cloud','tree','grass','water','sea','ocean','mountain'],
    'urban':  ['city','street','building','road','car','person','people'],
}

def extract_text_spatial(text):
    tokens = set(word_tokenize(text.lower()))
    spatial = np.array([sum(1 for w in kws if w in tokens)
                        for kws in SPATIAL_KEYWORDS.values()], dtype=float)
    if VADER_OK:
        scores = vader.polarity_scores(text)
        sentiment = np.array([scores['pos'], scores['neg'], scores['neu'], scores['compound']])
    else:
        sentiment = np.zeros(4)
    words = word_tokenize(text)
    n_words = len(words)
    n_unique = len(set(words))
    avg_len = np.mean([len(w) for w in words]) if words else 0
    text_stats = np.array([n_words, n_unique, avg_len,
                            n_unique/(n_words+1e-8)])
    return np.concatenate([spatial, sentiment, text_stats])

print('Stats texte spatiales...')
S_train = np.array(Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(extract_text_spatial)(t) for t in t_train))
S_val   = np.array(Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(extract_text_spatial)(t) for t in t_val))
S_test  = np.array(Parallel(n_jobs=N_JOBS, prefer='threads')(
    delayed(extract_text_spatial)(t) for t in t_test))

F_text_train = np.hstack([T_tfidf_train, S_train])
F_text_val   = np.hstack([T_tfidf_val,   S_val])
F_text_test  = np.hstack([T_tfidf_test,  S_test])
print(f'Shape features texte : {F_text_train.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6 — Features sémantiques : SentenceTransformer (si disponible)
# ═══════════════════════════════════════════════════════════════════
if SBERT_AVAILABLE:
    print('Encoding texte avec SentenceTransformer...')
    t0 = time.time()
    E_text_train = SBERT_MODEL.encode(list(t_train), batch_size=64,
                                       show_progress_bar=True, normalize_embeddings=True)
    E_text_val   = SBERT_MODEL.encode(list(t_val),   batch_size=64,
                                       show_progress_bar=True, normalize_embeddings=True)
    E_text_test  = SBERT_MODEL.encode(list(t_test),  batch_size=64,
                                       show_progress_bar=True, normalize_embeddings=True)
    print(f'  → {time.time()-t0:.1f}s, shape {E_text_train.shape}')
    F_text_train = np.hstack([F_text_train, E_text_train])
    F_text_val   = np.hstack([F_text_val,   E_text_val])
    F_text_test  = np.hstack([F_text_test,  E_text_test])
    print(f'Shape texte avec SBERT : {F_text_train.shape}')
else:
    print('SentenceTransformer non disponible — on continue sans embedding sémantique texte.')
    print('  → Installer avec: pip install sentence-transformers')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7 — Features visuelles : CLIP (si disponible)
# ═══════════════════════════════════════════════════════════════════
if CLIP_AVAILABLE:
    from PIL import Image as PILImage
    import torch

    @torch.no_grad()
    def extract_clip_image(paths, batch_size=32):
        all_feats = []
        for i in range(0, len(paths), batch_size):
            batch = []
            for p in paths[i:i+batch_size]:
                try:
                    img = PILImage.open(p).convert('RGB')
                    batch.append(CLIP_PREPROCESS(img))
                except Exception:
                    batch.append(torch.zeros(3, 224, 224))
            images = torch.stack(batch).to(DEVICE)
            feats = CLIP_MODEL.encode_image(images)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().numpy())
        return np.vstack(all_feats)

    @torch.no_grad()
    def extract_clip_text(texts, batch_size=256):
        all_feats = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            # CLIP tokenize tronque à 77 tokens
            tokens = clip.tokenize([t[:200] for t in batch], truncate=True).to(DEVICE)
            feats = CLIP_MODEL.encode_text(tokens)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().numpy())
        return np.vstack(all_feats)

    print('Extraction CLIP images...')
    t0 = time.time()
    C_img_train = extract_clip_image(p_train)
    C_img_val   = extract_clip_image(p_val)
    C_img_test  = extract_clip_image(p_test)
    print(f'  → {time.time()-t0:.1f}s, shape {C_img_train.shape}')

    print('Extraction CLIP textes...')
    t0 = time.time()
    C_txt_train = extract_clip_text(list(t_train))
    C_txt_val   = extract_clip_text(list(t_val))
    C_txt_test  = extract_clip_text(list(t_test))
    print(f'  → {time.time()-t0:.1f}s, shape {C_txt_train.shape}')

    # Similarité cosinus CLIP (feature scalaire très puissante)
    clip_sim_train = np.sum(C_img_train * C_txt_train, axis=1, keepdims=True)
    clip_sim_val   = np.sum(C_img_val   * C_txt_val,   axis=1, keepdims=True)
    clip_sim_test  = np.sum(C_img_test  * C_txt_test,  axis=1, keepdims=True)

    print(f'Similarité CLIP train — mean: {clip_sim_train.mean():.3f}, std: {clip_sim_train.std():.3f}')
    print(f'  cohérent:   {clip_sim_train[y_train==1].mean():.3f}')
    print(f'  incohérent: {clip_sim_train[y_train==0].mean():.3f}')

    # Ajouter CLIP aux features
    F_img_train = np.hstack([F_img_train, C_img_train, clip_sim_train])
    F_img_val   = np.hstack([F_img_val,   C_img_val,   clip_sim_val])
    F_img_test  = np.hstack([F_img_test,  C_img_test,  clip_sim_test])
    F_text_train = np.hstack([F_text_train, C_txt_train])
    F_text_val   = np.hstack([F_text_val,   C_txt_val])
    F_text_test  = np.hstack([F_text_test,  C_txt_test])
    print(f'\nShape finale img  : {F_img_train.shape}')
    print(f'Shape finale text : {F_text_train.shape}')
else:
    print('CLIP non disponible — fonctionnement en mode classique.')
    print('  → Pour activer: pip install git+https://github.com/openai/CLIP.git torch torchvision')
    clip_sim_train = np.zeros((len(t_train), 1))
    clip_sim_val   = np.zeros((len(t_val), 1))
    clip_sim_test  = np.zeros((len(t_test), 1))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 8 — Cross-features spatiaux texte ↔ image
# ═══════════════════════════════════════════════════════════════════
def build_cross_features_single(i, F_img, S_text, V_text_cca, V_img_cca):
    t  = V_text_cca[i]; v  = V_img_cca[i]; d  = t - v
    st = S_text[i];     fi = F_img[i]

    dist_cosine=float(cosine(t,v)); cos_sim=1-dist_cosine
    dist_euclidean=np.linalg.norm(d)
    dist_manhattan=float(cityblock(t,v)); dist_chebyshev=float(chebyshev(t,v))
    dist_canberra=float(canberra(t,v)); dist_braycurtis=float(braycurtis(t,v))
    dot_product=np.dot(t,v); angle=np.arccos(np.clip(cos_sim,-1,1))
    pearson_r=pearsonr(t,v)[0] if len(t)>2 else 0.0
    norm_ratio=np.linalg.norm(t)/(np.linalg.norm(v)+1e-8)
    proj_scalar=np.dot(t,v)/(np.linalg.norm(v)+1e-8)
    proj_vec=proj_scalar*v/(np.linalg.norm(v)+1e-8)
    orth_norm=np.linalg.norm(t-proj_vec)
    diff_std=d.std(); product=t*v; abs_diff=np.abs(d)

    IDX_V1=21; IDX_SPAT=IDX_V1+11; IDX_EXP=IDX_SPAT+6
    text_exp_top    = float(st[IDX_EXP+0]) if len(st)>IDX_EXP+2 else 0.0
    text_exp_bottom = float(st[IDX_EXP+1]) if len(st)>IDX_EXP+2 else 0.0
    text_has_left   = float(st[IDX_V1+0])  if len(st)>IDX_V1+4  else 0.0
    text_has_right  = float(st[IDX_V1+1])  if len(st)>IDX_V1+4  else 0.0

    N_HOG=81; N_SPAT=33; spat_start=-(N_HOG+N_SPAT); spat_end=-N_HOG if N_HOG>0 else None
    if len(fi)>N_HOG+N_SPAT:
        spat=fi[spat_start:spat_end]
        cy_norm=spat[0] if len(spat)>0 else 0.5; cx_norm=spat[1] if len(spat)>1 else 0.5
        asym_lr_canny=spat[4] if len(spat)>4 else 0.0
        asym_tb_canny=spat[5] if len(spat)>5 else 0.0
        top_hue=spat[11] if len(spat)>11 else 0.0; bot_hue=spat[12] if len(spat)>12 else 0.0
    else:
        cy_norm=cx_norm=0.5; asym_lr_canny=asym_tb_canny=0.0; top_hue=bot_hue=0.0

    cross_top_coherence    = text_exp_top    * (-asym_tb_canny)
    cross_bottom_coherence = text_exp_bottom * asym_tb_canny
    cross_left_right       = (text_has_left-text_has_right) * asym_lr_canny
    has_sky=int(any(w in ['sky','cloud','blue','sun','sunset','sunrise']
                   for w in word_tokenize(str(st).lower())))
    sky_hue_match=has_sky*(1-abs(top_hue-0.56))
    pos_coherence=(text_exp_bottom-text_exp_top)*(cy_norm-0.5)

    cross_feats=np.array([cross_top_coherence,cross_bottom_coherence,
                           cross_left_right,sky_hue_match,pos_coherence,
                           text_exp_top*top_hue,text_exp_bottom*bot_hue,
                           cx_norm,cy_norm])
    scalars=np.array([dist_cosine,cos_sim,dist_euclidean,dist_manhattan,
                       dist_chebyshev,dist_canberra,dist_braycurtis,
                       dot_product,angle,pearson_r,norm_ratio,proj_scalar,
                       orth_norm,diff_std,d.mean(),d.max(),d.min()])
    return np.concatenate([scalars,cross_feats,d,product,abs_diff])

def build_cross_spatial_features(F_img, S_text, V_text_cca, V_img_cca):
    N=len(F_img)
    results=Parallel(n_jobs=N_JOBS,prefer='threads')(
        delayed(build_cross_features_single)(i,F_img,S_text,V_text_cca,V_img_cca)
        for i in range(N))
    return np.array(results)

print('Cross-features définis.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 9 — Nettoyage + PCA + CCA
# ═══════════════════════════════════════════════════════════════════
def remove_correlated_features(X, threshold=0.95):
    X_s=X[:min(500,len(X))]; corr=np.corrcoef(X_s.T)
    np.fill_diagonal(corr,0); to_drop=set()
    for i in range(corr.shape[0]):
        if i in to_drop: continue
        for j in range(i+1,corr.shape[1]):
            if abs(corr[i,j])>threshold: to_drop.add(j)
    return [i for i in range(X.shape[1]) if i not in to_drop]

print('Nettoyage features...')
vt_img  = VarianceThreshold(threshold=1e-6)
vt_text = VarianceThreshold(threshold=1e-6)
F_img_train_vt  = vt_img.fit_transform(F_img_train)
F_img_val_vt    = vt_img.transform(F_img_val)
F_img_test_vt   = vt_img.transform(F_img_test)
F_text_train_vt = vt_text.fit_transform(F_text_train)
F_text_val_vt   = vt_text.transform(F_text_val)
F_text_test_vt  = vt_text.transform(F_text_test)

sc_img=StandardScaler(); sc_text=StandardScaler()
F_img_train_s  = sc_img.fit_transform(F_img_train_vt)
F_img_val_s    = sc_img.transform(F_img_val_vt)
F_img_test_s   = sc_img.transform(F_img_test_vt)
F_text_train_s = sc_text.fit_transform(F_text_train_vt)
F_text_val_s   = sc_text.transform(F_text_val_vt)
F_text_test_s  = sc_text.transform(F_text_test_vt)

keep_img=remove_correlated_features(F_img_train_s,threshold=0.95)
F_img_train_s=F_img_train_s[:,keep_img]
F_img_val_s  =F_img_val_s[:,keep_img]
F_img_test_s =F_img_test_s[:,keep_img]
print(f'Image après dédup : {len(keep_img)} features')

K_pre=128; K_cca=64
pca_img_pre  = PCA(n_components=min(K_pre,F_img_train_s.shape[1]),  random_state=42)
pca_text_pre = PCA(n_components=min(K_pre,F_text_train_s.shape[1]), random_state=42)
P_img_train  = pca_img_pre.fit_transform(F_img_train_s)
P_img_val    = pca_img_pre.transform(F_img_val_s)
P_img_test   = pca_img_pre.transform(F_img_test_s)
P_text_train = pca_text_pre.fit_transform(F_text_train_s)
P_text_val   = pca_text_pre.transform(F_text_val_s)
P_text_test  = pca_text_pre.transform(F_text_test_s)

print('CCA...')
cca=CCA(n_components=K_cca,max_iter=1000)
cca.fit(P_text_train,P_img_train)
V_text_train,V_img_train=cca.transform(P_text_train,P_img_train)
V_text_val,  V_img_val  =cca.transform(P_text_val,  P_img_val)
V_text_test, V_img_test =cca.transform(P_text_test, P_img_test)
for arr in [V_img_train,V_img_val,V_img_test,V_text_train,V_text_val,V_text_test]:
    arr[:]=normalize(arr)
print(f'Corrélations canoniques top-5:')
for i in range(5):
    r=np.corrcoef(V_text_train[:,i],V_img_train[:,i])[0,1]
    print(f'  C{i+1}: r={r:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 10 — Dataset final + sélection de features
# ═══════════════════════════════════════════════════════════════════
print('Construction dataset final...')
t0=time.time()
D_train=build_cross_spatial_features(F_img_train,S_train,V_text_train,V_img_train)
D_val  =build_cross_spatial_features(F_img_val,  S_val,  V_text_val,  V_img_val)
D_test =build_cross_spatial_features(F_img_test, S_test, V_text_test, V_img_test)
print(f'Shape dataset : {D_train.shape}  ({time.time()-t0:.1f}s)')

# Si CLIP dispo, ajouter la sim scalaire directement (très signal-dense)
if CLIP_AVAILABLE:
    D_train=np.hstack([D_train,clip_sim_train])
    D_val  =np.hstack([D_val,  clip_sim_val])
    D_test =np.hstack([D_test, clip_sim_test])

sc_dist=StandardScaler()
X_train=sc_dist.fit_transform(D_train)
X_val  =sc_dist.transform(D_val)
X_test =sc_dist.transform(D_test)

vt_final=VarianceThreshold(threshold=1e-4)
X_train=vt_final.fit_transform(X_train)
X_val  =vt_final.transform(X_val)
X_test =vt_final.transform(X_test)
print(f'Après VT : {X_train.shape[1]} features')

selector=SelectFromModel(
    RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1,
                            class_weight='balanced'),  # ← balanced ici aussi
    threshold='mean')
selector.fit(X_train,y_train)
X_train_sel=selector.transform(X_train)
X_val_sel  =selector.transform(X_val)
X_test_sel =selector.transform(X_test)
print(f'Après sélection : {X_train_sel.shape[1]} features')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 11 — Entraînement avec class_weight='balanced'
#
# Correction #1 du plafond v3 : les modèles ignoraient le déséquilibre.
# 'balanced' double le coût des erreurs sur la classe minoritaire.
# ═══════════════════════════════════════════════════════════════════
MODELS_V4 = {
    'LR balanced':     LogisticRegression(max_iter=2000, C=1.0,  random_state=42,
                                           n_jobs=-1, class_weight='balanced'),
    'LR C=0.1':        LogisticRegression(max_iter=2000, C=0.1,  random_state=42,
                                           n_jobs=-1, class_weight='balanced'),
    'LR C=5':          LogisticRegression(max_iter=2000, C=5.0,  random_state=42,
                                           n_jobs=-1, class_weight='balanced'),
    'RF balanced':     RandomForestClassifier(n_estimators=500, random_state=42,
                                              n_jobs=-1, class_weight='balanced'),
    'RF bal_sub':      RandomForestClassifier(n_estimators=500, random_state=42,
                                              n_jobs=-1, class_weight='balanced_subsample'),
    'SVM cal':         CalibratedClassifierCV(
                           LinearSVC(max_iter=5000, C=1.0, random_state=42,
                                     class_weight='balanced'), cv=3),
}

def eval_models_balanced(X_tr, X_vl, label=''):
    print(f'\n── {label} ──')
    print(f'{"Modèle":<22} {"CV3":>8} {"±":>5} {"ValAcc":>8} {"ValF1":>8}')
    print('-'*58)
    results={}
    skf=StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for name, mdl in MODELS_V4.items():
        m=copy.deepcopy(mdl)
        cv=cross_val_score(m, X_tr, y_train, cv=skf,
                            scoring='f1', n_jobs=-1)  # F1 comme métrique CV !
        m.fit(X_tr, y_train)
        yp=m.predict(X_vl)
        acc=accuracy_score(y_val, yp)
        f1=f1_score(y_val, yp)
        print(f'{name:<22} {cv.mean():.4f} {cv.std():.3f} {acc:.4f} {f1:.4f}')
        results[name]={'cv':cv.mean(),'acc':acc,'f1':f1,'model':m}
    return results

res_full = eval_models_balanced(X_train,     X_val,     'Dataset complet (balanced)')
res_sel  = eval_models_balanced(X_train_sel, X_val_sel, 'Dataset sélectionné (balanced)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 12 — Ajustement de seuil de décision
#
# Correction #2 du plafond v3 : le seuil 0.5 est trop conservateur
# pour une classe sous-représentée.
# On cherche le seuil qui maximise F1 sur la validation.
# ═══════════════════════════════════════════════════════════════════
all_res = {**res_full, **{n+'[sel]': v for n,v in res_sel.items()}}

# Modèle de base avec meilleur F1 val
best_base_name = max(all_res, key=lambda k: all_res[k]['f1'])
best_base = all_res[best_base_name]['model']
use_sel   = '[sel]' in best_base_name
X_v = X_val_sel if use_sel else X_val
X_tr_use = X_train_sel if use_sel else X_train
X_te_use = X_test_sel  if use_sel else X_test

print(f'Meilleur modèle base: {best_base_name}')

# Obtenir les probabilités (CalibratedClassifierCV le fait nativement)
if hasattr(best_base, 'predict_proba'):
    proba_val  = best_base.predict_proba(X_v)[:,1]
    proba_test = best_base.predict_proba(X_te_use)[:,1]
else:
    # Fallback : calibrer
    cal = CalibratedClassifierCV(best_base, cv='prefit')
    cal.fit(X_v, y_val)
    proba_val  = cal.predict_proba(X_v)[:,1]
    proba_test = cal.predict_proba(X_te_use)[:,1]

# Courbe precision-recall pour trouver le meilleur seuil
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, proba_val)
f1_scores = 2*precisions*recalls/(precisions+recalls+1e-8)

best_thresh_idx = np.argmax(f1_scores[:-1])
BEST_THRESHOLD  = thresholds_pr[best_thresh_idx]

print(f'\nSeuil optimal sur validation : {BEST_THRESHOLD:.4f}')
print(f'F1 @ seuil 0.5   : {f1_score(y_val, (proba_val>=0.5).astype(int)):.4f}')
print(f'F1 @ seuil opti  : {f1_scores[best_thresh_idx]:.4f}')
print(f'Acc @ seuil opti : {accuracy_score(y_val, (proba_val>=BEST_THRESHOLD).astype(int)):.4f}')

# Visualisation de la courbe
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(thresholds_pr, f1_scores[:-1], color='steelblue', lw=1.5)
axes[0].axvline(BEST_THRESHOLD, color='crimson', linestyle='--', label=f'optimal = {BEST_THRESHOLD:.3f}')
axes[0].axvline(0.5, color='gray', linestyle=':', label='seuil 0.5')
axes[0].set_xlabel('Seuil'); axes[0].set_ylabel('F1')
axes[0].set_title('F1 en fonction du seuil (val)'); axes[0].legend()

# Distribution des probabilités
axes[1].hist(proba_val[y_val==0], bins=40, alpha=0.5, color='salmon',  label='incohérent')
axes[1].hist(proba_val[y_val==1], bins=40, alpha=0.5, color='steelblue',label='cohérent')
axes[1].axvline(BEST_THRESHOLD, color='crimson',linestyle='--',label=f'seuil opt={BEST_THRESHOLD:.3f}')
axes[1].axvline(0.5, color='gray',linestyle=':',label='seuil 0.5')
axes[1].set_xlabel('P(cohérent)'); axes[1].set_ylabel('Nb exemples')
axes[1].set_title('Distribution des probabilités'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 13 — Stacking ensemble
#
# Correction #3 : combiner des modèles diversifiés réduit la variance
# et dépasse systématiquement le meilleur modèle individuel.
# ═══════════════════════════════════════════════════════════════════
print('Stacking ensemble...')

base_estimators = [
    ('rf',  RandomForestClassifier(n_estimators=400, random_state=42,
                                    n_jobs=-1, class_weight='balanced')),
    ('lr',  LogisticRegression(max_iter=2000, C=1.0, random_state=42,
                                n_jobs=-1, class_weight='balanced')),
    ('svm', CalibratedClassifierCV(
                LinearSVC(max_iter=5000, C=0.5, random_state=42,
                          class_weight='balanced'), cv=3)),
]

meta_learner = LogisticRegression(max_iter=1000, C=1.0, random_state=42)

stacker = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False,
)

t0 = time.time()
stacker.fit(X_tr_use, y_train)
print(f'  Stacking entraîné en {time.time()-t0:.1f}s')

yp_stack_val  = stacker.predict(X_v)
proba_stack_val  = stacker.predict_proba(X_v)[:,1]

# Seuil optimal pour le stacker
precisions_s, recalls_s, thresholds_s = precision_recall_curve(y_val, proba_stack_val)
f1s_s = 2*precisions_s*recalls_s/(precisions_s+recalls_s+1e-8)
best_idx_s = np.argmax(f1s_s[:-1])
THRESH_STACK = thresholds_s[best_idx_s]

yp_stack_val_opt = (proba_stack_val >= THRESH_STACK).astype(int)
acc_stack  = accuracy_score(y_val, yp_stack_val)
f1_stack   = f1_score(y_val, yp_stack_val)
acc_stack_opt = accuracy_score(y_val, yp_stack_val_opt)
f1_stack_opt  = f1_score(y_val, yp_stack_val_opt)

print(f'\nStacking @ seuil 0.5  — Acc: {acc_stack:.4f}  F1: {f1_stack:.4f}')
print(f'Stacking @ seuil opt {THRESH_STACK:.3f} — Acc: {acc_stack_opt:.4f}  F1: {f1_stack_opt:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 14 — Comparaison finale : tous modèles × tous seuils
# ═══════════════════════════════════════════════════════════════════
print('\n' + '═'*65)
print('  TABLEAU COMPARATIF COMPLET — Validation')
print('═'*65)
print(f'{"Modèle":<28} {"Acc@0.5":>8} {"F1@0.5":>8} {"Acc@opt":>8} {"F1@opt":>8} {"Seuil":>7}')
print('-'*65)

comparison_results = []

for name, res in all_res.items():
    m = res['model']
    X_v_use = X_val_sel if '[sel]' in name else X_val
    if hasattr(m, 'predict_proba'):
        proba = m.predict_proba(X_v_use)[:,1]
    else:
        try:
            proba = m.decision_function(X_v_use)
            proba = (proba - proba.min()) / (proba.max() - proba.min())
        except Exception:
            proba = np.full(len(y_val), 0.5)

    p, r, th = precision_recall_curve(y_val, proba)
    f1s = 2*p*r/(p+r+1e-8)
    bi = np.argmax(f1s[:-1])
    bt = th[bi] if len(th) > 0 else 0.5
    yp_opt = (proba >= bt).astype(int)

    acc05 = res['acc']; f105 = res['f1']
    acc_o = accuracy_score(y_val, yp_opt)
    f1_o  = f1_score(y_val, yp_opt)
    print(f'{name:<28} {acc05:>8.4f} {f105:>8.4f} {acc_o:>8.4f} {f1_o:>8.4f} {bt:>7.3f}')
    comparison_results.append({'name':name,'acc05':acc05,'f105':f105,'acc_opt':acc_o,'f1_opt':f1_o,'thresh':bt})

# Stacker
print(f'{"Stacker@0.5":<28} {acc_stack:>8.4f} {f1_stack:>8.4f} {acc_stack_opt:>8.4f} {f1_stack_opt:>8.4f} {THRESH_STACK:>7.3f}')
print('═'*65)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 15 — Évaluation TEST avec le meilleur modèle
# ═══════════════════════════════════════════════════════════════════

# Choisir: stacker ou meilleur individuel ?
best_val_f1_individual = max(r['f1_opt'] for r in comparison_results)
USE_STACKER = f1_stack_opt >= best_val_f1_individual

if USE_STACKER:
    FINAL_MODEL  = stacker
    FINAL_THRESH = THRESH_STACK
    FINAL_NAME   = 'Stacker (RF+LR+SVM) @ seuil opt'
    X_test_final = X_te_use
    proba_test   = FINAL_MODEL.predict_proba(X_test_final)[:,1]
else:
    best_row = max(comparison_results, key=lambda r: r['f1_opt'])
    FINAL_NAME   = best_row['name']
    FINAL_THRESH = best_row['thresh']
    FINAL_MODEL  = all_res[FINAL_NAME]['model']
    X_test_final = X_test_sel if '[sel]' in FINAL_NAME else X_test
    if hasattr(FINAL_MODEL, 'predict_proba'):
        proba_test = FINAL_MODEL.predict_proba(X_test_final)[:,1]
    else:
        proba_test = np.full(len(y_test), 0.5)

y_pred_test     = (proba_test >= FINAL_THRESH).astype(int)
y_pred_test_05  = (proba_test >= 0.5).astype(int)

test_acc    = accuracy_score(y_test, y_pred_test)
test_f1     = f1_score(y_test, y_pred_test)
test_acc_05 = accuracy_score(y_test, y_pred_test_05)
test_f1_05  = f1_score(y_test, y_pred_test_05)

print('═'*60)
print(f'  RÉSULTAT FINAL — Pipeline v4 Advanced')
print(f'  Modèle   : {FINAL_NAME}')
print(f'  Seuil    : {FINAL_THRESH:.4f}')
print('─'*60)
print(f'  Test Acc (seuil opt) : {test_acc:.4f}')
print(f'  Test F1  (seuil opt) : {test_f1:.4f}')
print(f'  Test Acc (seuil 0.5) : {test_acc_05:.4f}')
print(f'  Test F1  (seuil 0.5) : {test_f1_05:.4f}')
print('═'*60)
print(classification_report(y_test, y_pred_test,
      target_names=['incohérent','cohérent']))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 16 — Visualisations diagnostiques
# ═══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- Matrice de confusion (seuil optimal) ---
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['incohérent','cohérent'],
            yticklabels=['incohérent','cohérent'])
axes[0].set_title(f'Confusion — seuil opt ({FINAL_THRESH:.3f})')
axes[0].set_ylabel('Réel'); axes[0].set_xlabel('Prédit')

# --- Matrice de confusion (seuil 0.5) ---
cm05 = confusion_matrix(y_test, y_pred_test_05)
sns.heatmap(cm05, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['incohérent','cohérent'],
            yticklabels=['incohérent','cohérent'])
axes[1].set_title('Confusion — seuil 0.5 (baseline)')
axes[1].set_ylabel('Réel'); axes[1].set_xlabel('Prédit')

# --- ROC curve ---
fpr, tpr, _ = roc_curve(y_test, proba_test)
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, proba_test)
axes[2].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
axes[2].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].set_title('Courbe ROC — Test'); axes[2].legend()

plt.tight_layout(); plt.show()

print(f'\nAUC-ROC test : {auc:.4f}')
print(f'Recall cohérent  (seuil opt) : {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}')
print(f'Recall incohérent(seuil opt) : {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 17 — Analyse des erreurs résiduelles
#
# Comprendre ce qui reste difficile oriente la prochaine itération.
# ═══════════════════════════════════════════════════════════════════
errors = np.where(y_pred_test != y_test)[0]
correct = np.where(y_pred_test == y_test)[0]

print(f'Erreurs : {len(errors)} / {len(y_test)} ({100*len(errors)/len(y_test):.1f}%)')

# Analyse des faux positifs et faux négatifs
fp = np.where((y_pred_test==1) & (y_test==0))[0]  # prédit cohérent, réel incohérent
fn = np.where((y_pred_test==0) & (y_test==1))[0]  # prédit incohérent, réel cohérent
print(f'Faux positifs (prédit cohérent à tort) : {len(fp)}')
print(f'Faux négatifs (prédit incohérent à tort): {len(fn)}')

# Confiance sur les erreurs vs corrects
conf_errors  = np.abs(proba_test[errors]  - 0.5)
conf_correct = np.abs(proba_test[correct] - 0.5)
print(f'\nConfiance moyenne (erreurs)  : {conf_errors.mean():.4f}')
print(f'Confiance moyenne (corrects) : {conf_correct.mean():.4f}')
print(f'(Erreurs à haute confiance = biais systématique, pas d'incertitude)')

# Visualiser quelques exemples d'erreurs
def show_errors(indices, title, n=4):
    idx = indices[:n] if len(indices)>=n else indices
    if len(idx)==0:
        print(f'{title}: aucun exemple'); return
    fig, axes = plt.subplots(2, len(idx), figsize=(4*len(idx), 8))
    if len(idx)==1: axes=np.array([[axes[0]],[axes[1]]])
    for k, i in enumerate(idx):
        try:
            img = io.imread(p_test[i])
            if img.ndim==2: img=np.stack([img]*3,axis=-1)
            if img.shape[2]==4: img=img[:,:,:3]
            axes[0,k].imshow(img)
        except Exception:
            axes[0,k].text(0.5,0.5,'Image\nnon disponible',ha='center',va='center')
        axes[0,k].set_title(
            f'Réel:{["Inc","Coh"][y_test[i]]} / Prédit:{["Inc","Coh"][y_pred_test[i]]}\n'
            f'P(coh)={proba_test[i]:.3f}',
            fontsize=8, color='red')
        axes[0,k].axis('off')
        txt = t_test[i][:100]+'...' if len(t_test[i])>100 else t_test[i]
        axes[1,k].text(0.5,0.5,txt,ha='center',va='center',wrap=True,fontsize=7)
        axes[1,k].axis('off')
    plt.suptitle(title,fontsize=11,fontweight='bold')
    plt.tight_layout(); plt.show()

# Erreurs à haute confiance (les plus problématiques)
high_conf_fp = fp[np.argsort(proba_test[fp])[::-1]][:4]
high_conf_fn = fn[np.argsort(proba_test[fn])][:4]

show_errors(high_conf_fp, 'Faux positifs confiants (cohérent prédit, incohérent réel)')
show_errors(high_conf_fn, 'Faux négatifs confiants (incohérent prédit, cohérent réel)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 18 — Importance des features
# ═══════════════════════════════════════════════════════════════════
# Extraire l'importance du meilleur modèle individuel RF
rf_model = None
for name, res in all_res.items():
    if 'RF' in name and hasattr(res['model'],'feature_importances_'):
        if rf_model is None or res['f1']>all_res[list(all_res.keys())[0]]['f1']:
            rf_model = res['model']
            rf_name  = name

if rf_model is not None:
    imp = rf_model.feature_importances_
    top_k   = min(30, len(imp))
    top_idx = np.argsort(imp)[::-1][:top_k]

    plt.figure(figsize=(16, 5))
    plt.bar(range(top_k), imp[top_idx], color='steelblue')
    plt.xticks(range(top_k), [f'f{i}' for i in top_idx], rotation=45, ha='right', fontsize=8)
    plt.title(f'Top-30 features — {rf_name}')
    plt.tight_layout(); plt.show()

    scalar_names = [
        'dist_cosine','cos_sim','dist_euclidean','dist_manhattan','dist_chebyshev',
        'dist_canberra','dist_braycurtis','dot_product','angle','pearson_r',
        'norm_ratio','proj_scalar','orth_norm','diff_std','diff_mean','diff_max','diff_min',
        'cross_top','cross_bottom','cross_left_right','sky_hue_match','pos_coherence',
        'txt_exp_top*top_hue','txt_exp_bot*bot_hue','cx_norm','cy_norm',
    ]
    if CLIP_AVAILABLE:
        scalar_names.append('clip_sim')
    print(f'Top-15 features ({rf_name}):')
    for rank, idx in enumerate(top_idx[:15]):
        name_f = scalar_names[idx] if idx<len(scalar_names) else f'cca_dim_{idx-len(scalar_names)}'
        print(f'  {rank+1:2d}. [{idx:4d}] {name_f:<30} imp={imp[idx]:.6f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 19 — Sauvegarde + rapport synthétique
# ═══════════════════════════════════════════════════════════════════
import joblib

joblib.dump(FINAL_MODEL,   'v4_best_model.pkl')
joblib.dump(sc_dist,       'v4_scaler_dist.pkl')
joblib.dump(sc_img,        'v4_scaler_img.pkl')
joblib.dump(sc_text,       'v4_scaler_text.pkl')
joblib.dump(pca_img_pre,   'v4_pca_img.pkl')
joblib.dump(pca_text_pre,  'v4_pca_text.pkl')
joblib.dump(cca,           'v4_cca.pkl')
joblib.dump(tfidf,         'v4_tfidf.pkl')
joblib.dump(vt_img,        'v4_vt_img.pkl')
joblib.dump(vt_text,       'v4_vt_text.pkl')
joblib.dump(selector,      'v4_selector.pkl')
joblib.dump(vt_final,      'v4_vt_final.pkl')
joblib.dump({'threshold': FINAL_THRESH}, 'v4_threshold.pkl')

print('Sauvegarde OK.')
print()
print('━'*60)
print('  RAPPORT FINAL — Pipeline v4 Advanced')
print('━'*60)
print(f'  Modèle final : {FINAL_NAME}')
print(f'  Seuil opt    : {FINAL_THRESH:.4f}')
print(f'  Test Acc     : {test_acc:.4f}  (était ~0.66 en v3)')
print(f'  Test F1      : {test_f1:.4f}')
print(f'  AUC-ROC      : {auc:.4f}')
print('─'*60)
print('  Améliorations apportées :')
print('  1. class_weight="balanced" → recall cohérent +X%')
print('  2. Seuil ajusté sur PR curve → F1 optimisé')
print('  3. Stacking RF+LR+SVM → variance réduite')
if SBERT_AVAILABLE:
    print('  4. SentenceTransformer → features sémantiques texte')
if CLIP_AVAILABLE:
    print('  5. CLIP → similarité sémantique image-texte')
print('─'*60)
print('  Pour aller plus loin :')
print('  - BLIP-2 / LLaVA pour features multimodales avancées')
print('  - Fine-tuning CLIP sur le dataset (si GPU)')
print('  - Data augmentation (flip, crop, paraphrase)')
print('  - Calibration Isotonic Regression')
print('━'*60)